In [ ]:
import os
import sys
import json
import hashlib
import subprocess

print(f"Python: {sys.version}")
print("=" * 60)

# ============================================================
# Instalar dependencias si es necesario
# ============================================================
print("[INFO] Verificando dependencias...")
try:
    import huggingface_hub
    print(f"[OK] huggingface_hub ya instalado: {huggingface_hub.__version__}")
except ImportError:
    print("[INFO] Instalando huggingface_hub...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.24.0"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"[ERROR] pip install huggingface_hub: {result.stderr[-300:]}")
    else:
        print("[OK] huggingface_hub instalado")

try:
    import huggingface_hub
    print(f"[OK] huggingface_hub version: {huggingface_hub.__version__}")
except ImportError as e:
    print(f"[ERROR] No se pudo instalar huggingface_hub: {e}")
    huggingface_hub = None

# ============================================================
# Configuración
# ============================================================
MODELS = {
    "qwen3_vl_2b": {
        "repo_id": "Qwen/Qwen3-VL-2B-Instruct",
        "local_dir": "/tmp/kaggle_models/qwen3vl_2b"
    },
    "qwen3_vl_4b": {
        "repo_id": "Qwen/Qwen3-VL-4B-Instruct",
        "local_dir": "/tmp/kaggle_models/qwen3vl_4b"
    }
}

MANIFEST_PATH = "/tmp/kaggle_models/manifest.json"

# ============================================================
# Función: Descargar modelo completo desde Hugging Face
# ============================================================
def download_model(repo_id, local_dir, model_name):
    os.makedirs(local_dir, exist_ok=True)
    print(f"\n[{model_name}] Descargando repositorio {repo_id}...")
    
    if huggingface_hub is None:
        print(f"[ERROR] {model_name}: huggingface_hub no disponible")
        return None
    
    try:
        # Usar snapshot_download para descargar todo el repositorio
        result = huggingface_hub.snapshot_download(
            repo_id=repo_id,
            local_dir=local_dir,
            local_dir_use_symlinks=False,
            allow_patterns=[".safetensors", ".json", "config", "tokenizer", "tokenizer.model", "tokenizer.json", "special_tokensmap.json", "generation_config.json", "modeling", "modeling_*", "modeling_qwen", "modeling_qwen3", "configuration", "configuration_qwen", "configuration_qwen3", "tokenizer", "tokenizer.model", "pretrained_cfg.yaml", "README.md", "LICENSE", "que_", "*.json", "*.yaml"]
        )
        print(f"[OK] {model_name}: {local_dir}")
        
        downloaded_files = []
        for root, dirs, files in os.walk(local_dir):
            for f in files:
                fp = os.path.join(root, f)
                size = os.path.getsize(fp)
                downloaded_files.append({"path": fp, "size_bytes": size})
        
        if not downloaded_files:
            print(f"[ERROR] {model_name}: Ningún archivo encontrado en {local_dir}")
            return None
        
        for df in downloaded_files:
            with open(df["path"], "rb") as fh:
                df["sha256"] = hashlib.sha256(fh.read()).hexdigest()
        
        total_size = sum(d["size_bytes"] for d in downloaded_files)
        print(f"[OK] {model_name}: {total_size / 1024 / 1024:.2f} MB total")
        return downloaded_files
    
    except Exception as e:
        print(f"[ERROR] {model_name}: {e}")
        import traceback
        traceback.print_exc()
        return None

# ============================================================
# Función: Crear manifiesto
# ============================================================
def create_manifest():
    manifest = {"files": [], "generated_at": str(os.environ.get('KAGGLE_KERNEL_RUN_DATE', 'unknown'))}
    
    for model_key, model_cfg in MODELS.items():
        local_dir = model_cfg["local_dir"]
        if not os.path.exists(local_dir):
            print(f"[ADVERTENCIA] {model_key} no encontrado: {local_dir}")
            continue
        for root, dirs, files in os.walk(local_dir):
            for f in files:
                fp = os.path.join(root, f)
                size = os.path.getsize(fp)
                with open(fp, "rb") as fh:
                    sha256 = hashlib.sha256(fh.read()).hexdigest()
                manifest["files"].append({
                    "model": model_key,
                    "repo_id": MODELS[model_key]["repo_id"],
                    "filename": f,
                    "path": fp,
                    "size_bytes": size,
                    "size_mb": round(size / 1024 / 1024, 2),
                    "sha256": sha256
                })
    
    with open(MANIFEST_PATH, "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"\n[OK] Manifiesto guardado en {MANIFEST_PATH}")
    return manifest

# ============================================================
# Bloque principal
# ============================================================
print("=" * 60)
print("Downloads de Qwen3-VL")
print("=" * 60)

results = {}

try:
    results["qwen3_vl_2b"] = download_model(
        repo_id=MODELS["qwen3_vl_2b"]["repo_id"],
        local_dir=MODELS["qwen3_vl_2b"]["local_dir"],
        model_name="Qwen3-VL-2B-Instruct"
    )
except Exception as e:
    print(f"[ERROR] Qwen3-VL-2B: {e}")
    import traceback
    traceback.print_exc()

try:
    results["qwen3_vl_4b"] = download_model(
        repo_id=MODELS["qwen3_vl_4b"]["repo_id"],
        local_dir=MODELS["qwen3_vl_4b"]["local_dir"],
        model_name="Qwen3-VL-4B-Instruct"
    )
except Exception as e:
    print(f"[ERROR] Qwen3-VL-4B: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "=" * 60)
print("Generando manifiesto")
print("=" * 60)
manifest = create_manifest()

print("\n" + "=" * 60)
print("RESUMEN FINAL")
print("=" * 60)

for model_key, files in results.items():
    if files:
        total = sum(f["size_bytes"] for f in files)
        print(f"[{model_key}] {len(files)} archivo(s), {total / 1024 / 1024:.2f} MB")
        for f in files[:5]:
            print(f"  - {f['filename']}: {f['size_bytes'] / 1024 / 1024:.2f} MB, sha256={f['sha256'][:16]}...")
        if len(files) > 5:
            print(f"  ... y {len(files) - 5} archivos más")
    else:
        print(f"[{model_key}] FALLÓ")

print(f"\nTotal de archivos en manifiesto: {len(manifest.get('files', []))}")

print("\n[INFO] Si este notebook se ejecuta en Kaggle, los archivos en /tmp serán preservados")
